# Test Both Paths Side-by-Side

You have deployed the AgentCore Gateway (Path B) and synced the Registry catalog into it. Now it is time to run real requests through both paths and compare the experience.

**Path A** goes through CloudFront and NGINX to Docker-based MCP servers. It is fast and simple.

**Path B** goes through the AgentCore Gateway, which validates JWT tokens, runs request and response interceptors, and dispatches to targets. It supports Lambda and HTTP targets that NGINX cannot serve.

In this notebook you will:
1. Set up URLs and authentication for both paths
2. Call a tool through Path A (NGINX)
3. Call the same tool through Path B (AgentCore Gateway)
4. Call a Lambda-only tool that only Path B can serve (search-knowledge-base)
5. Compare the results and discuss the tradeoffs

In [ ]:
%pip install httpx==0.27.0 --quiet

## Step 1: Setup -- Gather URLs and Authenticate

We need:
- **CloudFront URL** (Path A entry point) -- from the Module 3 CFN export
- **Registry URL** -- to look up tool metadata
- **AgentCore Gateway URL** (Path B entry point) -- from the AgentCore control plane API
- **Registry API token** -- for Path A (Registry/NGINX authentication), from the `workshop-RegistryApiTokenSecretArn` export
- **Cognito JWT** -- for Path B (AgentCore Gateway requires a CUSTOM_JWT authorizer token)

In [ ]:
import boto3, json, httpx, time

region = boto3.session.Session().region_name or "us-west-2"

# --- CloudFormation exports (Module 3) ---
cfn = boto3.client("cloudformation", region_name=region)
exports = {}
for page in cfn.get_paginator("list_exports").paginate():
    for e in page["Exports"]:
        exports[e["Name"]] = e["Value"]

def get_export(primary, fallback=None):
    """Look up a workshop-* export, falling back to mcp-gateway-* if needed."""
    val = exports.get(primary)
    if val:
        return val
    if fallback:
        return exports.get(fallback, "")
    return ""

REGISTRY_URL = get_export("workshop-RegistryUrl")
CLOUDFRONT_URL = get_export("workshop-MainCloudFrontUrl", "mcp-gateway-MainCloudFrontUrl")

# --- AgentCore Gateway URL (Module 4) ---
agentcore = boto3.client("bedrock-agentcore-control", region_name=region)
gateways = agentcore.list_gateways().get("items", [])
gw = [g for g in gateways if g["name"] == "tools-gateway"]
if not gw:
    raise RuntimeError("Gateway 'tools-gateway' not found! Did you run notebook 02?")

GATEWAY_ID = gw[0]["gatewayId"]
GATEWAY_URL = f"https://{GATEWAY_ID}.gateway.bedrock-agentcore.{region}.amazonaws.com/mcp"

# --- Path A: Registry API token (accepted by Registry auth layer / NGINX) ---
sm = boto3.client("secretsmanager", region_name=region)
api_token_arn = exports.get("workshop-RegistryApiTokenSecretArn", "")
if not api_token_arn:
    raise RuntimeError(
        "Export 'workshop-RegistryApiTokenSecretArn' not found. "
        "Is workshop-registry-stack deployed and CREATE_COMPLETE?"
    )
api_secret = json.loads(
    sm.get_secret_value(SecretId=api_token_arn)["SecretString"]
)
STATIC_TOKEN = api_secret.get("api_token", "")
PATH_A_HEADERS = {"Authorization": f"Bearer {STATIC_TOKEN}"}

# --- Path B: Cognito JWT (required by AgentCore Gateway CUSTOM_JWT authorizer) ---
m2m_secret_arn = get_export("workshop-CognitoM2MClientSecretArn", "mcp-gateway-CognitoM2MClientSecretArn")
cognito_domain = get_export("workshop-CognitoDomain", "mcp-gateway-CognitoDomain")

if not (m2m_secret_arn and cognito_domain):
    raise RuntimeError(
        "Cannot acquire Cognito M2M token: "
        f"workshop-CognitoM2MClientSecretArn={m2m_secret_arn!r}, "
        f"workshop-CognitoDomain={cognito_domain!r}. "
        "Verify Module 3 (workshop-registry-stack) is CREATE_COMPLETE."
    )
m2m_json = json.loads(sm.get_secret_value(SecretId=m2m_secret_arn)["SecretString"])
token_url = f"https://{cognito_domain}.auth.{region}.amazoncognito.com/oauth2/token"
token_resp = httpx.post(
    token_url,
    headers={"Content-Type": "application/x-www-form-urlencoded"},
    auth=(m2m_json["client_id"], m2m_json["client_secret"]),
    data={"grant_type": "client_credentials", "scope": "mcp-servers-unrestricted/read mcp-servers-unrestricted/execute"},
    timeout=10,
)
if token_resp.status_code != 200:
    raise RuntimeError(
        f"Cognito token endpoint returned {token_resp.status_code}: {token_resp.text[:300]}"
    )
GATEWAY_TOKEN = token_resp.json()["access_token"]
print("  Cognito JWT acquired (value not printed)")

PATH_B_HEADERS = {"Authorization": f"Bearer {GATEWAY_TOKEN}"}

print("Setup complete:")
print(f"  CloudFront URL (Path A):  {CLOUDFRONT_URL}")
print(f"  Registry URL:             {REGISTRY_URL}")
print(f"  Gateway URL (Path B):     {GATEWAY_URL}")
print(f"  Gateway ID:               {GATEWAY_ID}")
print(f"  Path A auth:              Registry API token")
print(f"  Path B auth:              Cognito JWT")

## Step 2: Path A -- Access the Registry via NGINX

Path A routes through CloudFront to NGINX. The Registry API is the primary service available via this path. Docker-based MCP servers (like the flights and hotels tools) are also served via NGINX.

However, **Lambda-backed tools cannot be served via Path A** -- NGINX cannot route `lambda://` URLs. This is the key limitation that Path B solves.

The request flow is: **Notebook -> CloudFront -> ALB -> NGINX -> Registry API**.

In [ ]:
# Path A: List all tools via the Registry API (served through NGINX)

print("PATH A: Listing tools via Registry API (CloudFront -> NGINX -> FastAPI)")
print(f"  URL: {REGISTRY_URL}/api/servers")
print("=" * 60)

t0 = time.time()
resp_a = httpx.get(
    f"{REGISTRY_URL}/api/servers",
    headers=PATH_A_HEADERS,
    timeout=15,
)
elapsed_a = (time.time() - t0) * 1000

print(f"  Status: {resp_a.status_code} ({elapsed_a:.0f}ms)")

if resp_a.status_code == 200:
    servers = resp_a.json()
    if isinstance(servers, dict):
        servers = servers.get("servers", servers.get("items", []))
    print(f"  Servers found: {len(servers)}")
    print()
    for s in servers:
        name = s.get("display_name") or s.get("server_name") or s.get("name", "?")
        proxy = s.get("proxy_pass_url", "?")
        is_lambda = "lambda://" in str(proxy)
        path_a_status = "PATH A ONLY" if not is_lambda else "PATH B ONLY (lambda://)"
        print(f"    {name:30s}  [{path_a_status}]")
else:
    print(f"  Error: {resp_a.text[:200]}")

print()
print("Note: Tools with lambda:// URLs can ONLY be called through Path B.")
print("NGINX cannot invoke Lambda functions -- it is a reverse proxy for HTTP services.")

## Step 3: Path B -- List Tools via the AgentCore Gateway

Now we make the same `tools/list` call, but through the AgentCore Gateway. The request flow is:

**Notebook -> AgentCore Gateway -> Request Interceptor (audit log) -> Target -> Response Interceptor (guardrails) -> Back to notebook**

The gateway multiplexes all targets behind a single endpoint. We specify which target to reach using the target name.

In [ ]:
# Path B: List tools via the AgentCore Gateway
# The gateway target name follows the "tg-{server_name}" convention from the Sync Lambda

# First, list the available targets so we know what names to use
targets = agentcore.list_gateway_targets(gatewayIdentifier=GATEWAY_ID).get("items", [])
target_names = [t["name"] for t in targets]

print("Available gateway targets:")
for name in target_names:
    print(f"  - {name}")

# Pick the flights-mcp target (synced from the MCP server)
target_name = "tg-workshop-flights-mcp"
path_b_url = f"{GATEWAY_URL}"

print(f"\nPATH B: Listing tools via AgentCore Gateway")
print(f"  URL: {path_b_url}")
print(f"  Target: {target_name}")
print("=" * 60)

# AgentCore Gateway dispatches by the namespaced tool name ("{target}___{tool}")
# returned from tools/list. No routing header is needed.

t0 = time.time()
resp_b_list = httpx.post(
    path_b_url,
    headers={
        **PATH_B_HEADERS,
        "Content-Type": "application/json",
    },
    json={
        "jsonrpc": "2.0",
        "id": 1,
        "method": "tools/list",
        "params": {},
    },
    timeout=30,
)
elapsed_b_list = (time.time() - t0) * 1000

print(f"  Status: {resp_b_list.status_code} ({elapsed_b_list:.0f}ms)")
if resp_b_list.status_code != 200:
    raise RuntimeError(
        f"Gateway tools/list returned {resp_b_list.status_code}: {resp_b_list.text[:300]}"
    )
result = resp_b_list.json()
if "error" in result:
    raise RuntimeError(f"Gateway tools/list JSON-RPC error: {result['error']}")
tools = result.get("result", {}).get("tools", [])
print(f"  Tools found: {len(tools)}")
print()
print("  Note: Tool names are namespaced by the gateway as '{target}___{tool}'.")
print("  Use the full namespaced name when calling tools/call -- that is how")
print("  the gateway decides which target to dispatch the request to.")
for t in tools:
    print(f"    - {t['name']}: {t.get('description', '')[:60]}")

## Step 4: Path B -- Call a Tool Through the Gateway

Let's invoke a specific tool through Path B. We will call `search_flights` on the flights-mcp target. This request goes through the full interceptor chain: the request interceptor logs the call, the target executes it, and the response interceptor can apply guardrails.

In [ ]:
# Path B: Call a tool through the AgentCore Gateway
# The gateway namespaces tool names as "{target}___{tool_name}"
# Use the full namespaced name from the tools/list response -- the namespace prefix
# is what tells the gateway which target to dispatch to.

NAMESPACED_TOOL = "tg-workshop-flights-mcp___search_flights"

print(f"PATH B: Calling '{NAMESPACED_TOOL}' via AgentCore Gateway")
print("=" * 60)

t0 = time.time()
resp_b_call = httpx.post(
    path_b_url,
    headers={
        **PATH_B_HEADERS,
        "Content-Type": "application/json",
    },
    json={
        "jsonrpc": "2.0",
        "id": 2,
        "method": "tools/call",
        "params": {
            "name": NAMESPACED_TOOL,
            "arguments": {"origin": "SFO", "destination": "TYO", "date": "2026-09-15"},
        },
    },
    timeout=30,
)
elapsed_b_call = (time.time() - t0) * 1000

print(f"  Status: {resp_b_call.status_code} ({elapsed_b_call:.0f}ms)")
if resp_b_call.status_code != 200:
    raise RuntimeError(
        f"Gateway tools/call returned {resp_b_call.status_code}: {resp_b_call.text[:300]}"
    )
result = resp_b_call.json()
if "error" in result:
    raise RuntimeError(f"Gateway tools/call JSON-RPC error: {result['error']}")
print(f"  Response:")
print(json.dumps(result, indent=2))

## Step 5: Gateway-Only Tools (Lambda Targets)

Some tools are only available through Path B. The `search-knowledge-base` tool registered in notebook 03 uses a `lambda://` URL, which means NGINX has no way to route to it. Only the AgentCore Gateway can invoke Lambda targets natively.

This is a key advantage of Path B: it extends the tool catalog beyond what Docker containers can provide.

In [ ]:
# Identify gateway-only targets (Lambda and external HTTP targets)
# These are the tools that NGINX cannot serve

print("Gateway-Only Targets (Path B exclusive):")
print("=" * 60)

gateway_only = []
for t in targets:
    name = t.get("name", "")
    # Lambda targets and external HTTP targets are gateway-only
    if "search-knowledge-base" in name or "weather-api" in name:
        gateway_only.append(t)

if gateway_only:
    for t in gateway_only:
        print(f"  Target: {t['name']}")
        print(f"    Status: {t.get('status', '?')}")
        print(f"    Target ID: {t.get('targetId', '?')}")
        print()
else:
    print("  No gateway-only targets found.")
    print("  Did you register the Lambda and OpenAPI tools in notebook 03?")

# Try calling the knowledge base search tool through Path B
kb_target = "tg-search-knowledge-base"
KB_NAMESPACED_TOOL = "tg-search-knowledge-base___search-knowledge-base"

if kb_target in target_names:
    print(f"\nCalling '{kb_target}' via Path B (this is a Lambda-backed MCP tool):")
    print("-" * 60)

    t0 = time.time()
    resp_kb = httpx.post(
        path_b_url,
        headers={
            **PATH_B_HEADERS,
            "Content-Type": "application/json",
        },
        json={
            "jsonrpc": "2.0",
            "id": 3,
            "method": "tools/call",
            "params": {
                "name": KB_NAMESPACED_TOOL,
                "arguments": {"query": "shipping policy", "max_results": 3},
            },
        },
        timeout=30,
    )
    elapsed_kb = (time.time() - t0) * 1000
    print(f"  Status: {resp_kb.status_code} ({elapsed_kb:.0f}ms)")
    print(f"  Response: {resp_kb.text[:300]}")
else:
    print(f"\n  Target '{kb_target}' not found. Run notebooks 03 and 04 first.")

## Comparison: Path A vs. Path B

You just ran the same tool through both paths. Here is what you should observe:

| Aspect | Path A (NGINX) | Path B (AgentCore Gateway) |
|---|---|---|
| **Latency** | Faster (single NGINX hop) | Slower (interceptor Lambdas add overhead) |
| **Authentication** | NGINX auth_request to Cognito | Gateway custom authorizer (Cognito JWT) |
| **Audit trail** | NGINX access log only | Structured audit log from request interceptor |
| **Content screening** | None | Response interceptor can apply Bedrock Guardrails |
| **Lambda tools** | Not supported | Native Lambda invoke |
| **External APIs** | Not supported | HTTP target with governance |

**The audit trail is the key difference.** Every request through Path B is logged by the request interceptor Lambda with structured metadata: who called what tool, when, with which arguments. This is not visible in the HTTP response -- it is written to CloudWatch Logs. You can verify it in the AWS Console under the `agentcore-gateway-request-interceptor` log group.

**Next:** In notebook 06, you will add Bedrock Guardrails to the response interceptor, making Path B even more valuable for regulated workloads.